# 01 Structural Breaks and the COVID Problem

Walk-forward backtests measure average error across regimes. Sometimes the regimes are so different that 'average error' is the wrong concept. This notebook is about detecting that — and what to do once you have.


## Table of Contents
- [What is a structural break](#what-is-break)
- [Visual evidence on macro data](#visual)
- [Chow test on a known break](#chow)
- [Coefficients pre- vs post-break](#coefficients)
- [Forecasting through a break](#forecasting)
- [Strategies for breaks](#strategies)
- [Checkpoint (Self-Check)](#checkpoint-self-check)
- [Solutions (Reference)](#solutions-reference)


## Why This Notebook Matters
Most macro models look great until something genuinely unprecedented arrives. The classic example is Q2 2020: real GDP collapsed at an annualized rate that had no precedent in the post-WWII data. A model fit on 1985–2019 data is being asked to extrapolate to a regime it never saw.

There are two distinct failure modes:
1. The **outliers** themselves are wildly atypical (Q2 2020 GDP).
2. The **relationships** between variables changed (the slope of GDP growth on the yield-curve spread is different post-COVID).

This notebook gives you tools for both: visual diagnostics, the Chow test for a known break, and a few strategies to keep modeling sensibly when a break has happened.

## Prerequisites (Quick Self-Check)
- §02 (regression) — comfortable with OLS coefficient interpretation.
- §02b (ML regression) helps but is not required.
- The previous notebook (`00_walk_forward_backtest.ipynb`).

## What You Will Produce
- A Chow-test verdict for at least two candidate break points.
- A side-by-side comparison of pre- vs post-break OLS coefficients.
- A demonstration of pre-COVID model failing on COVID-era data, with at least one mitigation strategy applied.

## Common Pitfalls
- Searching over many break dates and reporting the smallest p-value as if you only tested one. That is multiple testing.
- Using a Chow test when you don't have a clear hypothesis about *where* the break is. Use a CUSUM or Quandt-Andrews test for unknown break dates (out of scope here).
- Dropping COVID quarters as a reflex. Sometimes the right move; sometimes you are throwing out exactly the data that informs deployment.
- Treating a crisis dummy as a 'fix' when it is really a 'we don't know what to do here.'

## Quick Fixes (When You Get Stuck)
- If `chow_test` errors that a sub-sample is too small, check the break_index — Chow needs more than $k$ observations on each side.
- If you cannot find a break visually, plot the residuals of a single OLS fit over time. Big swings in residuals around a date suggest a break.

## Matching Guide
- `docs/guides/04_honest_forecasting/01_structural_breaks_and_covid.md`


<a id="environment-bootstrap"></a>
## Environment Bootstrap


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    p = start
    for _ in range(8):
        if (p / 'src').exists() and (p / 'docs').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root. Start Jupyter from the repo root.')


PROJECT_ROOT = find_repo_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
SAMPLE_DIR = DATA_DIR / 'sample'

PROJECT_ROOT


<a id="what-is-break"></a>
## What is a structural break

A structural break is a point in time after which the *parameters* of a model change. In a regression $y_t = \alpha + \beta x_t + u_t$, that could mean:
- the intercept changes (the average level of $y$ shifts),
- the slope changes (the response of $y$ to $x$ is different),
- both,
- the noise variance changes.

If you fit one model across a break, you get a parameter estimate that is some weighted average of the two regimes. The fit will be worse than either regime alone, *and* extrapolating it forward will systematically miss in whichever regime is currently active.

Common candidate break dates in macro data:
- **1984**: 'Great Moderation' — volatility of US output drops.
- **2008Q3**: collapse of Lehman Brothers / global financial crisis.
- **2020Q1**: COVID-19 onset.


<a id="visual"></a>
## Visual evidence on macro data


### Your Turn (1): Load the panel and plot GDP growth


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

path = PROCESSED_DIR / 'macro_quarterly.csv'
if path.exists():
    df = pd.read_csv(path, index_col=0, parse_dates=True)
else:
    df = pd.read_csv(SAMPLE_DIR / 'macro_quarterly_sample.csv', index_col=0, parse_dates=True)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df.index, df['gdp_growth_qoq'])
for date, label in [('2008-09-30', 'GFC'), ('2020-03-31', 'COVID')]:
    ax.axvline(pd.Timestamp(date), color='red', linestyle='--', alpha=0.6)
    ax.text(pd.Timestamp(date), ax.get_ylim()[1] * 0.9, ' ' + label, color='red')
ax.set_title('Quarterly GDP growth')
ax.set_ylabel('% QoQ')
plt.tight_layout()


### Your Turn (2): Plot the OLS residuals over time

Fit a single OLS across the whole sample and plot the residuals. Big swings around a date are a hint that the relationship has changed.


In [ ]:
import statsmodels.api as sm

y_col = 'gdp_growth_qoq'
x_cols = ['T10Y2Y_lag1', 'UNRATE_lag1', 'FEDFUNDS_lag1']
x_cols = [c for c in x_cols if c in df.columns]

panel = df[[y_col] + x_cols].dropna().copy()
X = sm.add_constant(panel[x_cols], has_constant='add')
res_full = sm.OLS(panel[y_col], X).fit()

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(panel.index, res_full.resid)
ax.axhline(0, color='k', linewidth=0.5)
for date in ['2008-09-30', '2020-03-31']:
    ax.axvline(pd.Timestamp(date), color='red', linestyle='--', alpha=0.6)
ax.set_title('OLS residuals: gdp_growth_qoq ~ macro lags')
ax.set_ylabel('residual (pp)')
plt.tight_layout()


<a id="chow"></a>
## Chow test on a known break

The Chow test asks: do the OLS coefficients from the pre-break sub-sample equal those from the post-break sub-sample? It is an F-test comparing the residual sum of squares of the pooled fit to the sum of the two sub-sample fits.

$$
F = \frac{(RSS_{pool} - (RSS_{pre} + RSS_{post})) / k}{(RSS_{pre} + RSS_{post}) / (n - 2k)}
$$

Reject (small p) → coefficients differ; you have a break.

**Caveat:** Chow assumes you specified the break date *a priori*. Searching over many candidate dates is multiple testing and inflates false-positive rates. There are tests for unknown break dates (Quandt-Andrews) but they are out of scope here.


### Your Turn (1): Test the COVID break


In [ ]:
from src import econometrics

covid_break = pd.Timestamp('2020-03-31')
res_chow_covid = econometrics.chow_test(
    panel, y_col=y_col, x_cols=x_cols, break_index=covid_break
)
print('Chow at 2020-Q1:', res_chow_covid.as_dict())


### Your Turn (2): Test the GFC break


In [ ]:
gfc_break = pd.Timestamp('2008-09-30')
# TODO: run chow_test with break_index=gfc_break.
res_chow_gfc = ...
print('Chow at 2008-Q3:', res_chow_gfc.as_dict())


### Your Turn (3): Interpretation

Compare the two p-values. Is the same model 'broken' by both breaks, by one, or by neither?

Write 4–6 sentences. Hint: the GFC was a big event in GDP *levels* but the relationship between predictors and growth may not have shifted persistently. COVID is more likely to have shifted the relationship because the shock was on a different mechanism (lockdowns, supply chains).


In [ ]:
notes = """
...
"""
print(notes)


<a id="coefficients"></a>
## Coefficients pre- vs post-break

If the Chow test flagged a break, fit OLS separately on pre and post sub-samples and look at the coefficients. The pattern is informative even when individual coefficients are noisy.


### Your Turn (1): Compare pre- and post-COVID coefficients


In [ ]:
pre = panel.loc[panel.index < covid_break]
post = panel.loc[panel.index >= covid_break]

res_pre = sm.OLS(pre[y_col], sm.add_constant(pre[x_cols], has_constant='add')).fit()
res_post = sm.OLS(post[y_col], sm.add_constant(post[x_cols], has_constant='add')).fit()

comparison = pd.DataFrame({
    'pre_covid': res_pre.params,
    'post_covid': res_post.params,
    'change': res_post.params - res_pre.params,
}).round(4)
comparison


<a id="forecasting"></a>
## Forecasting through a break

The point of detecting a break is to make better decisions, not just to write down a p-value. Here is the realistic question: a model trained on pre-2020 data — how badly does it fail on 2020 and after?


### Your Turn (1): Train pre-COVID, forecast COVID era


In [ ]:
from src.evaluation import regression_metrics

X_pre = sm.add_constant(pre[x_cols], has_constant='add')
X_post = sm.add_constant(post[x_cols], has_constant='add')

model_pre = sm.OLS(pre[y_col], X_pre).fit()
yhat_pre_on_post = model_pre.predict(X_post)

m = regression_metrics(post[y_col].to_numpy(), yhat_pre_on_post.to_numpy())
print('Pre-COVID model evaluated on post-COVID:', {k: round(v, 4) for k, v in m.items()})


<a id="strategies"></a>
## Strategies for breaks

Once you have evidence of a break, the question is what to do. There is no single right answer; the right answer depends on what you are forecasting and how much post-break data you have.

Three reasonable strategies:

1. **Crisis dummy.** Add an indicator variable that is 1 during the break period (e.g., 2020Q1–2020Q4) and 0 otherwise. This soaks up the level shift but assumes coefficients on other variables are unchanged.
2. **Drop the break period.** Drop the most extreme quarters (e.g., 2020Q1–2020Q2) and refit. This is honest about the fact that the model has nothing useful to say about that period.
3. **Refit on post-break data only.** Throw away the pre-break data and refit. Works only if you have enough post-break data.

Each strategy has different tradeoffs. Pick one based on what you are using the model for; do not stack them all in hopes of a smaller RMSE.


### Your Turn (1): Strategy 1 — crisis dummy


In [ ]:
panel_dummy = panel.copy()
panel_dummy['covid_dummy'] = (
    (panel_dummy.index >= '2020-03-31') & (panel_dummy.index <= '2020-12-31')
).astype(int)

x_cols_d = x_cols + ['covid_dummy']
X_d = sm.add_constant(panel_dummy[x_cols_d], has_constant='add')
res_dummy = sm.OLS(panel_dummy[y_col], X_d).fit()
print(res_dummy.summary().tables[1])


### Your Turn (2): Strategy 2 — drop the break quarters


In [ ]:
drop_window = ('2020-03-31', '2020-12-31')
mask = (panel.index < drop_window[0]) | (panel.index > drop_window[1])
panel_dropped = panel.loc[mask]

# TODO: refit OLS on panel_dropped and compare its coefficients to the full-sample fit (res_full).
X_drop = sm.add_constant(panel_dropped[x_cols], has_constant='add')
res_dropped = ...
...


### Your Turn (3): Choose your default and write your reasoning

Write 6–8 sentences:
- Which strategy did you pick as default and why?
- What does each strategy commit you to (in terms of model assumptions)?
- If you had to ship one of these to a stakeholder this week, which would it be?


In [ ]:
decision = """
...
"""
print(decision)


<a id="checkpoint-self-check"></a>
## Checkpoint (Self-Check)


In [ ]:
# Sanity asserts (uncomment after the cells above run cleanly):
# assert res_chow_covid.pvalue < 0.10  # weak evidence at minimum on the bundled sample
# assert res_chow_gfc.pvalue > res_chow_covid.pvalue  # GFC less Chow-detectable than COVID on this sample
# assert 'covid_dummy' in res_dummy.params.index
...


## Extensions (Optional)
- Run a CUSUM (cumulative sum of recursive residuals) plot using `statsmodels.stats.diagnostic.recursive_olsresiduals`. It detects breaks at *unknown* dates.
- Repeat the Chow analysis on a different target — say `INDPRO_lag1`'s relationship with GDP growth.
- Compare the post-COVID OOS RMSE under each of the three strategies on a small holdout window. Which one wins?
- Apply the COVID-dummy strategy inside a walk-forward backtest. Does it improve walk-forward RMSE compared to the no-dummy version from the previous notebook?


## Reflection
- A Chow p-value tells you 'something changed' but not 'what should I do.' Did you find a strategy you would defend in a meeting?
- The next section flips the question entirely: instead of asking 'what predicts y?' it asks 'what *causes* y?' Why is that a different question, and which one have you been answering so far?


<a id="solutions-reference"></a>
## Solutions (Reference)

<details><summary>Solution: Chow at 2008-Q3</summary>

```python
res_chow_gfc = econometrics.chow_test(
    panel, y_col=y_col, x_cols=x_cols, break_index=gfc_break
)
```

</details>

<details><summary>Solution: Drop the break quarters</summary>

```python
X_drop = sm.add_constant(panel_dropped[x_cols], has_constant='add')
res_dropped = sm.OLS(panel_dropped[y_col], X_drop).fit()
comparison = pd.DataFrame({
    'full_sample': res_full.params,
    'covid_dropped': res_dropped.params,
}).round(4)
comparison
```

Expected pattern: the covid_dropped coefficients are typically *closer* to the pre-COVID coefficients than the full-sample ones. That is the pull from the COVID outliers being removed.

</details>

<details><summary>Solution: Strategy choice</summary>

There is no single right answer. A defensible default for forecasting on quarterly macro data:
- if you must produce a single number for next quarter and have stakeholders, use the **crisis dummy** so the model can still leverage all data;
- if you are doing a backtest or analysis, **drop the break quarters** because including them distorts coefficients in ways that complicate interpretation;
- only **refit on post-break data only** if you have at least 30+ post-break observations and a reason to believe the structural change is permanent.

</details>
